# Baseline Benchmark

Runs the initial LLM call on all benchmark queries, evaluates the results, and saves everything to disk (including `prompt.txt` per query so a refined run can load it later).

In [1]:
import importlib
import evaluate as evaluate_module
import analysis as analysis_module
from evaluate import evaluate, print_metrics, print_evaluation_summary
from benchmarks import load_benchmarks
from llm import call_llm, build_prompt
from analysis import analyze
import output as output_module

## Configuration

In [2]:
from pathlib import Path
BENCHMARK_DIR = Path("./benchmark")
BENCHMARK_TYPES = ["socbenchd_1"]  # add more benchmark types here
BENCHMARK_LIMIT = None  # set to None for all sectors
QUERY_LIMIT = 50        # set to None for all queries
MAX_WORKERS = 10
MODEL = "deepseek-ai/DeepSeek-V4-Pro"

benchmark_sets, total_available, total_queries_available = load_benchmarks(BENCHMARK_DIR, BENCHMARK_TYPES, BENCHMARK_LIMIT, None)
total_queries = min(QUERY_LIMIT, total_queries_available) if QUERY_LIMIT is not None else total_queries_available
print(f"Loaded {len(benchmark_sets)} benchmark sets (total available: {total_available})")
print(f"Total queries: {total_queries} (of {total_queries_available} available)")
print(f"Workers: {MAX_WORKERS} | Model: {MODEL}")

Loaded 11 benchmark sets (total available: 11)
Total queries: 50 (of 110 available)
Workers: 10 | Model: deepseek-ai/DeepSeek-V4-Pro


## Prompt Template

In [3]:
PROMPT_TEMPLATE = '''You're doing a Service Composition.
You are given a set of REST API specifications and a task description.
Your job is to write Python code using the appropriate client library that fulfills the task by calling the necessary endpoints in the correct order. Import requests and create a function called compose.

Rules:
- Use the requests library.
- Only use endpoints defined in the provided specifications.
- Return ONLY raw Python code. No markdown, no code fences, no comments, no notes, no explanations — nothing but the code itself.
- Import the requests library and create a function called compose, where all the requests shall be called. Do NOT call that function.

## Task
{query}

## Source
{services_block}

'''


## Initial LLM Call

In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed

tasks = []
for benchmark in benchmark_sets:
    if QUERY_LIMIT is not None and len(tasks) >= QUERY_LIMIT:
        break
    for query_index, query in enumerate(benchmark['queries'], start=1):
        if QUERY_LIMIT is not None and len(tasks) >= QUERY_LIMIT:
            break
        tasks.append((benchmark, query_index, query))

def _call_initial(args):
    benchmark, query_index, query = args
    prompt = build_prompt(benchmark['services'], query['query'], PROMPT_TEMPLATE)
    generated = call_llm(prompt, MODEL, '')
    generated += '\n\ncompose()'
    return {
        'query_index': query_index,
        'sector_name': benchmark['name'],
        'query': query,
        'prompt': prompt,
        'generated': generated,
        'service_files': benchmark.get('service_files', []),
        'model': MODEL
    }

sector_results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(_call_initial, t): t for t in tasks}
    for future in as_completed(futures):
        result = future.result()
        sector_results.append(result)
        print(f"[{len(sector_results)}/{total_queries}] [{result['sector_name']}] Query {result['query_index']} done")

sector_results.sort(key=lambda r: (r['sector_name'], r['query_index']))

[1/50] [01-energy] Query 2 done
[2/50] [01-energy] Query 8 done
[3/50] [01-energy] Query 5 done
[4/50] [01-energy] Query 1 done
[5/50] [01-energy] Query 6 done
[6/50] [01-energy] Query 10 done
[7/50] [01-energy] Query 9 done
[8/50] [01-energy] Query 7 done
[9/50] [01-energy] Query 3 done
[10/50] [01-energy] Query 4 done
[11/50] [02-materials] Query 3 done
[12/50] [02-materials] Query 5 done
[13/50] [02-materials] Query 7 done
[14/50] [02-materials] Query 2 done
[15/50] [02-materials] Query 6 done
[16/50] [02-materials] Query 4 done
[17/50] [02-materials] Query 1 done
[18/50] [03-industrials] Query 4 done
[19/50] [03-industrials] Query 2 done
[20/50] [03-industrials] Query 5 done
[21/50] [03-industrials] Query 3 done
[22/50] [03-industrials] Query 1 done
[23/50] [02-materials] Query 9 done
[24/50] [03-industrials] Query 7 done
[25/50] [04-consumer discretionary] Query 2 done
[26/50] [04-consumer discretionary] Query 1 done
[27/50] [03-industrials] Query 9 done
[28/50] [03-industrials] Q

## Evaluate

In [5]:
for result in sector_results:
    initial_metrics = evaluate(result['generated'], result['query'].get('endpoints', []))
    result['initial_metrics'] = initial_metrics
    print_metrics(initial_metrics, f"Initial evaluation - Query {result['query_index']}")

Initial evaluation - Query 1
  Precision: 1.00
  Recall:    1.00
  F1:        1.00
  Extracted: ['GET /alerts', 'GET /equipment-status', 'GET /weather-impact-analysis', 'POST /alert-settings', 'POST /renewable/integration/status']
  Expected:  ['GET /alerts', 'GET /equipment-status', 'GET /weather-impact-analysis', 'POST /alert-settings', 'POST /renewable/integration/status']
  Missing:   []
  Extra:     []
Initial evaluation - Query 2
  Precision: 1.00
  Recall:    1.00
  F1:        1.00
  Extracted: ['GET /carbon-emissions', 'GET /prediction-summary', 'GET /resources/status', 'POST /report-feedback', 'POST /smart-meters/data']
  Expected:  ['GET /carbon-emissions', 'GET /prediction-summary', 'GET /resources/status', 'POST /report-feedback', 'POST /smart-meters/data']
  Missing:   []
  Extra:     []
Initial evaluation - Query 3
  Precision: 0.75
  Recall:    0.75
  F1:        0.75
  Extracted: ['GET /electricity-demand', 'GET /energy-patterns', 'GET /real-time-data', 'GET /reports/mon

## Save Outputs

In [6]:
from datetime import datetime
importlib.reload(output_module)
run_name = datetime.now().strftime('%Y-%m-%d_%H-%M-%S') + "_baseline"
outdir = output_module.make_output_dir(run_name)
for r in sector_results:
    output_module.write_query_output(outdir, r)

run_config = {
    'benchmark_dir': str(BENCHMARK_DIR),
    'benchmark_types': BENCHMARK_TYPES,
    'benchmark_limit': BENCHMARK_LIMIT,
    'query_limit': QUERY_LIMIT,
    'baseline': True,
    'model': MODEL,
}
output_module.write_overall_summary(outdir, sector_results, run_config)
print('Wrote outputs to', outdir)

Wrote outputs to output/2026-06-16_15-57-18_baseline


## Summary

In [7]:
print_evaluation_summary([result['initial_metrics'] for result in sector_results], "Initial Evaluation Summary")

Initial Evaluation Summary
  Average Precision: 0.53
  Average Recall:    0.63
  Average F1:        0.56
  Avg. Missing Endpoints: 1.60
  Avg. Extra Endpoints:   2.78
  Correct Compositions: 5/50 (10.0%)
